# 02 — Target Construction & Validation

Build the (1 / 0) training labels from the corrosion observation dates, following the Airbus convention. Steps:

1. Load the two training files and **sort by date**.
2. Drop corrosion rows for aircraft with **no environment history at all** (keep an aircraft even if env is missing for the *exact* date, as long as it has env for *some* date).
3. For each corrosion row: add **y = 1** at the observation month, and **y = 0** at the month `OFFSET_MONTHS` before it.
4. For each **y = 0** row, check whether the same aircraft has a corrosion observation *before* that date — **flag** it (do not delete).

See target definition in [`Docs/problem-analysis.md` §5](../Docs/problem-analysis.md).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)

# y=0 reference point: months before the observation date.
# 24 = official Airbus / Kaggle convention (see problem-analysis.md §5).
OFFSET_MONTHS = 24

DATA = Path('..') / 'data' / 'raw'

## 1. Load the training files and sort by date

In [2]:
corr = pd.read_csv(DATA / 'corrosions_training.csv', parse_dates=['observation_date'])
env  = pd.read_csv(DATA / 'environment_training.csv', parse_dates=['month_start_date'])

# Sort by date
corr = corr.sort_values('observation_date').reset_index(drop=True)
env  = env.sort_values(['aircraft_id', 'month_start_date']).reset_index(drop=True)

print('corrosions_training:', corr.shape)
print('environment_training:', env.shape)
corr.head()

corrosions_training: (790, 4)
environment_training: (63524, 36)


,observation_date,aircraft_delivery_year,aircraft_delivery_month,aircraft_id
0,2016-10-19,2015,2,e8e7d4
1,2016-12-07,2015,6,3329fc
2,2017-03-20,2015,8,0c8a7b
3,2017-05-10,2016,12,cd7110
4,2017-06-19,2015,7,ae7d89


## 2. Drop corrosion rows for aircraft with no environment history
Only aircraft that appear **nowhere** in `environment_training` are removed. An aircraft that has env data for *some* month (just not the exact reference date) is kept.

In [3]:
aircraft_with_env = set(env['aircraft_id'].unique())
n_before = len(corr)
corr = corr[corr['aircraft_id'].isin(aircraft_with_env)].reset_index(drop=True)
n_after = len(corr)
print(f'corrosion aircraft before: {n_before}')
print(f'corrosion aircraft after : {n_after}')
print(f'dropped (no env history) : {n_before - n_after}')

corrosion aircraft before: 790
corrosion aircraft after : 758
dropped (no env history) : 32


## 3. Build the (1 / 0) labels
For each corrosion row → **y = 1** at the observation month and **y = 0** at `OFFSET_MONTHS` before it. Month arithmetic uses pandas `Period[M]`.

In [4]:
corr['obs_month'] = corr['observation_date'].dt.to_period('M')

pos = pd.DataFrame({'aircraft_id': corr['aircraft_id'],
                    'ref_month':   corr['obs_month'],
                    'y': 1})
neg = pd.DataFrame({'aircraft_id': corr['aircraft_id'],
                    'ref_month':   corr['obs_month'] - OFFSET_MONTHS,
                    'y': 0})

labels = pd.concat([pos, neg], ignore_index=True)
labels['year_month'] = labels['ref_month'].astype(str)  # 'YYYY-MM' -> matches env.year_month
labels = labels.sort_values(['aircraft_id', 'ref_month']).reset_index(drop=True)

print('label rows:', len(labels))
labels['y'].value_counts()

label rows: 1516


y
0    758
1    758
Name: count, dtype: int64

In [5]:
labels.head(8)

,aircraft_id,ref_month,y,year_month
0,002eab,2023-04,0,2023-04
1,002eab,2025-04,1,2025-04
2,009e95,2020-04,0,2020-04
3,009e95,2022-04,1,2022-04
4,01c05c,2023-01,0,2023-01
5,01c05c,2025-01,1,2025-01
6,02884d,2019-03,0,2019-03
7,02884d,2021-03,1,2021-03


## 4. Is environment data available at each reference month?
We kept aircraft even when env is missing for the exact date — so mark which label rows actually have a matching `(aircraft_id, year_month)` row in `environment_training`. Informative for feature-building later; rows are **not** dropped.

In [6]:
env_keys = env[['aircraft_id', 'year_month']].drop_duplicates()
env_keys['env_available'] = True
labels = labels.merge(env_keys, on=['aircraft_id', 'year_month'], how='left')
labels['env_available'] = labels['env_available'].fillna(False).astype(bool)

labels.groupby('y')['env_available'].value_counts().rename('rows')

y  env_available
0  True             654
   False            104
1  True             616
   False            142
Name: rows, dtype: int64

## 5. Flag y = 0 rows that follow an earlier corrosion (same aircraft)
For each **y = 0** row, check whether the same aircraft has any corrosion observation in a month **strictly before** the reference month. If so, calling that month 'healthy' is questionable — so we **flag** it. We do **not** delete these rows.

> Note: `corrosions_training` has one observation per aircraft, and the y=0 month is by construction *before* it, so we expect **zero** flags here — this is a guard that will catch the problem automatically if multiple observations per aircraft ever appear.

In [7]:
# month(s) of observed corrosion per aircraft
corr_months = (corr.groupby('aircraft_id')['obs_month']
                   .apply(lambda s: sorted(s.tolist())).to_dict())

def has_prior_corrosion(row):
    if row['y'] != 0:
        return False
    months = corr_months.get(row['aircraft_id'], [])
    return any(m < row['ref_month'] for m in months)

labels['flag_prior_corrosion'] = labels.apply(has_prior_corrosion, axis=1)
print('y=0 rows flagged (prior corrosion):', int(labels['flag_prior_corrosion'].sum()))

y=0 rows flagged (prior corrosion): 0


## 6. Summary of flagged rows
Surface every flag at the end so nothing is silently dropped.

In [8]:
print('Total label rows        :', len(labels))
print('  y = 1                 :', int((labels.y == 1).sum()))
print('  y = 0                 :', int((labels.y == 0).sum()))
print('env unavailable at ref  :', int((~labels.env_available).sum()))
print('flag_prior_corrosion    :', int(labels.flag_prior_corrosion.sum()))

flagged = labels[labels['flag_prior_corrosion'] | ~labels['env_available']]
print('\nrows with any flag:', len(flagged))
flagged.head(20)

Total label rows        : 1516
  y = 1                 : 758
  y = 0                 : 758
env unavailable at ref  : 246
flag_prior_corrosion    : 0

rows with any flag: 246


,aircraft_id,ref_month,y,year_month,env_available,flag_prior_corrosion
28,06f4d3,2020-12,0,2020-12,False,False
36,077ee1,2016-10,0,2016-10,False,False
42,08ce08,2018-01,0,2018-01,False,False
45,0932c4,2025-10,1,2025-10,False,False
48,09b5d6,2022-05,0,2022-05,False,False
49,09b5d6,2024-05,1,2024-05,False,False
55,0bdf39,2025-11,1,2025-11,False,False
56,0c708c,2024-01,0,2024-01,False,False
57,0c708c,2026-01,1,2026-01,False,False
60,0c8a7b,2015-03,0,2015-03,False,False


## 7. Persist the labeled table
Saved to `output/` with a timestamp, per project conventions. This feeds [`03_eda_drivers.ipynb`](03_eda_drivers.ipynb).

In [9]:
out_dir = Path('..') / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
out_path = out_dir / f'labels_{ts}.csv'
labels.to_csv(out_path, index=False)
print('saved:', out_path)
labels.head()

saved: ../output/labels_20260611_141930.csv


,aircraft_id,ref_month,y,year_month,env_available,flag_prior_corrosion
0,002eab,2023-04,0,2023-04,True,False
1,002eab,2025-04,1,2025-04,True,False
2,009e95,2020-04,0,2020-04,True,False
3,009e95,2022-04,1,2022-04,True,False
4,01c05c,2023-01,0,2023-01,True,False
